# Session 3 — SQL for Data Investigators
## Steering Wheel Manufacturing · DuckDB Query Notebook

> **Before running:** `python generate_steering_wheel_data.py`

| Station | Parquet file | Scope |
|---|---|---|
| Molding Skelet | `station_molding.parquet` | All — PT55 / PT66 / PT77 |
| Quality Check | `station_quality_check.parquet` | All |
| Foaming Spuma | `station_foaming.parquet` | All |
| Conductor Incalzire | `station_conductor.parquet` | PT66 + PT77 |
| Laser Bombardment | `station_laser.parquet` | PT77 only |
| Tapitat Piele | `station_tapitat.parquet` | All |
| Materials Stock | `materials_stock.parquet` | Reference |
| Materials Log | `materials_log.parquet` | Consumption events |

**How to use:** Each section has a business question in plain English → SQL → your interpretation.

---
## Setup

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect()

DATA_DIR = "data"
TABLES = {
    "materials_stock":       f"{DATA_DIR}/materials_stock.parquet",
    "materials_log":         f"{DATA_DIR}/materials_log.parquet",
    "station_molding":       f"{DATA_DIR}/station_molding.parquet",
    "station_quality_check": f"{DATA_DIR}/station_quality_check.parquet",
    "station_foaming":       f"{DATA_DIR}/station_foaming.parquet",
    "station_conductor":     f"{DATA_DIR}/station_conductor.parquet",
    "station_laser":         f"{DATA_DIR}/station_laser.parquet",
    "station_tapitat":       f"{DATA_DIR}/station_tapitat.parquet",
}

for name, path in TABLES.items():
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_parquet('{path}')")

print("All 8 tables registered.")

In [ ]:
def q(sql, rows=15):
    """Run SQL → return Pandas DataFrame. rows=-1 returns full result."""
    df = con.execute(sql).df()
    return df if rows == -1 else df.head(rows)

---
## 1. Explore the Schema

Before writing any query: **what columns do I have?**

In [ ]:
q("DESCRIBE station_molding", rows=-1)

In [ ]:
# Row count for every table
sql = """
SELECT 'station_molding'       AS tbl, COUNT(*) AS rows FROM station_molding
UNION ALL SELECT 'station_quality_check', COUNT(*) FROM station_quality_check
UNION ALL SELECT 'station_foaming',       COUNT(*) FROM station_foaming
UNION ALL SELECT 'station_conductor',     COUNT(*) FROM station_conductor
UNION ALL SELECT 'station_laser',         COUNT(*) FROM station_laser
UNION ALL SELECT 'station_tapitat',       COUNT(*) FROM station_tapitat
UNION ALL SELECT 'materials_stock',       COUNT(*) FROM materials_stock
UNION ALL SELECT 'materials_log',         COUNT(*) FROM materials_log
ORDER BY rows DESC
"""
q(sql, rows=-1)

---
## 2. Single-Table Queries
`SELECT` · `WHERE` · `GROUP BY` · `ORDER BY`

### Q1 — Product type distribution + scrap rate at molding
> *How many units per product type, and what is the scrap rate for each?*

In [ ]:
# This multiline string variable named `sql` contains a complete SQL query.
# The query is written for beginners to read and is meant to be sent to a database
# (in this case, probably DuckDB) to summarize information stored in a table
# called `station_molding`.
#
# The goal of the query:
#   For each different type of product (`product_type`), calculate:
#     - how many total records/units there are
#     - how many have each possible `cycle_result` value: 'OK', 'REWORK', 'SCRAP'
#     - what percentage of them are 'SCRAP'
#
# Let's go through the query line by line:

sql = """
SELECT
    product_type,
    COUNT(*)                                                              AS total_units,
    SUM(CASE WHEN cycle_result = 'OK'     THEN 1 ELSE 0 END)              AS ok_count,
    SUM(CASE WHEN cycle_result = 'REWORK' THEN 1 ELSE 0 END)              AS rework_count,
    SUM(CASE WHEN cycle_result = 'SCRAP'  THEN 1 ELSE 0 END)              AS scrap_count,
    ROUND(100.0 * SUM(CASE WHEN cycle_result = 'SCRAP' THEN 1 ELSE 0 END)
                / COUNT(*), 2)                                             AS scrap_pct
FROM station_molding
GROUP BY product_type
ORDER BY product_type
"""
q(sql, rows=-1)

### Q2 — Which shift scraps the most at molding?
> *If Night shift has a higher scrap rate, suspect temperature drop or fatigue.*

In [ ]:
sql = """
SELECT
    shift,
    COUNT(*)                                                            AS total,
    SUM(CASE WHEN cycle_result = 'SCRAP' THEN 1 ELSE 0 END)             AS scraps,
    ROUND(AVG(temperature_c), 2)                                         AS avg_temp_c,
    ROUND(AVG(humidity_pct),  1)                                         AS avg_humidity_pct,
    ROUND(100.0 * SUM(CASE WHEN cycle_result = 'SCRAP' THEN 1 ELSE 0 END)
                / COUNT(*), 2)                                           AS scrap_pct
FROM station_molding
GROUP BY shift
ORDER BY scrap_pct DESC
"""
q(sql, rows=-1)

### Q3 — Is foaming hitting the SIC volume targets?
> *SIC: PT55 = 8.0 ml | PT66 = 10.0 ml | PT77 = 12.0 ml (±0.5 ml tolerance)*

In [ ]:
sql = """
SELECT
    product_type,
    COUNT(*)                                                    AS total,
    ROUND(AVG(foam_volume_ml),    3)                             AS avg_ml,
    ROUND(STDDEV(foam_volume_ml), 3)                             AS stddev_ml,
    ROUND(MIN(foam_volume_ml),    2)                             AS min_ml,
    ROUND(MAX(foam_volume_ml),    2)                             AS max_ml,
    SUM(CASE WHEN foam_result = 'OK'        THEN 1 ELSE 0 END)  AS ok,
    SUM(CASE WHEN foam_result = 'UNDERFILL' THEN 1 ELSE 0 END)  AS underfill,
    SUM(CASE WHEN foam_result = 'OVERFILL'  THEN 1 ELSE 0 END)  AS overfill,
    SUM(CASE WHEN foam_result = 'SCRAP'     THEN 1 ELSE 0 END)  AS scrap
FROM station_foaming
GROUP BY product_type
ORDER BY product_type
"""
q(sql, rows=-1)

---
## 3. Aggregation + HAVING

`HAVING` = `WHERE` for aggregated groups. It filters **after** `GROUP BY`.

### Q4 — Which operators have the worst scrap rate? (min 200 cycles)
> `HAVING COUNT(*) >= 200` removes small-sample noise.

In [ ]:
sql = """
SELECT
    operator_id,
    COUNT(*)                                                              AS total_cycles,
    SUM(CASE WHEN cycle_result = 'SCRAP' THEN 1 ELSE 0 END)               AS scraps,
    ROUND(100.0 * SUM(CASE WHEN cycle_result = 'SCRAP' THEN 1 ELSE 0 END)
                / COUNT(*), 2)                                             AS scrap_pct,
    ROUND(AVG(temperature_c), 2)                                           AS avg_temp_c,
    ROUND(AVG(pressure_bar),  3)                                           AS avg_pressure_bar
FROM station_molding
GROUP BY operator_id
HAVING COUNT(*) >= 200
ORDER BY scrap_pct DESC
LIMIT 10
"""
q(sql, rows=-1)

### Q5 — Batches with conductor resistance violations
> *A single bad batch here often traces back to a supplier wire lot.*

In [ ]:
sql = """
SELECT
    batch_id,
    product_type,
    COUNT(*)                                                             AS total_in_batch,
    SUM(CASE WHEN resistance_ohm >= 2.5 THEN 1 ELSE 0 END)               AS violations,
    ROUND(AVG(resistance_ohm), 3)                                         AS avg_ohm,
    ROUND(MAX(resistance_ohm), 3)                                         AS worst_ohm,
    ROUND(100.0 * SUM(CASE WHEN resistance_ohm >= 2.5 THEN 1 ELSE 0 END)
                / COUNT(*), 1)                                            AS violation_pct
FROM station_conductor
GROUP BY batch_id, product_type
HAVING SUM(CASE WHEN resistance_ohm >= 2.5 THEN 1 ELSE 0 END) > 0
ORDER BY violations DESC
LIMIT 15
"""
q(sql, rows=-1)

### Q6 — Inspectors with highest scrap flag rate (min 500 inspections)

In [ ]:
sql = """
SELECT
    inspector_id,
    COUNT(*)                                                              AS total,
    SUM(CASE WHEN overall_result = 'PASS'   THEN 1 ELSE 0 END)            AS pass_count,
    SUM(CASE WHEN overall_result = 'REWORK' THEN 1 ELSE 0 END)            AS rework_count,
    SUM(CASE WHEN overall_result = 'SCRAP'  THEN 1 ELSE 0 END)            AS scrap_count,
    ROUND(100.0 * SUM(CASE WHEN overall_result = 'SCRAP' THEN 1 ELSE 0 END)
                / COUNT(*), 2)                                             AS scrap_flag_pct
FROM station_quality_check
GROUP BY inspector_id
HAVING COUNT(*) >= 500
ORDER BY scrap_flag_pct DESC
"""
q(sql, rows=-1)

---
## 4. Multi-Table JOINs

All station tables share `product_id`. Use it to trace a unit across stations.

```
station_molding  ─┐
station_foaming  ─┤  JOIN on product_id
station_conductor─┤
station_laser    ─┘
```

### Q7 — Products that PASSED QC but had a foam failure
> *QC said OK → foaming said not OK. The problem is at the foaming station, not the skeleton.*

In [ ]:
sql = """
SELECT
    q.product_type,
    COUNT(*)                                                         AS affected_units,
    SUM(CASE WHEN f.foam_result = 'UNDERFILL' THEN 1 ELSE 0 END)     AS underfill,
    SUM(CASE WHEN f.foam_result = 'OVERFILL'  THEN 1 ELSE 0 END)     AS overfill,
    SUM(CASE WHEN f.foam_result = 'SCRAP'     THEN 1 ELSE 0 END)     AS scrap,
    ROUND(AVG(f.foam_volume_ml), 3)                                   AS avg_foam_ml
FROM station_quality_check q
JOIN station_foaming f ON q.product_id = f.product_id
WHERE q.overall_result = 'PASS'
  AND f.foam_result    != 'OK'
GROUP BY q.product_type
ORDER BY affected_units DESC
"""
q(sql, rows=-1)

### Q8 — PT77 First-Pass Yield across all 6 stations
> *How many PT77 units passed every single station? This is the key KPI.*

In [ ]:
sql = """
WITH pt77_full AS (
    SELECT
        m.product_id,
        m.cycle_result        AS molding,
        qc.overall_result     AS quality_check,
        f.foam_result         AS foaming,
        c.result              AS conductor,
        l.result              AS laser,
        t.result              AS tapitat
    FROM station_molding m
    JOIN station_quality_check qc ON m.product_id = qc.product_id
    JOIN station_foaming f        ON m.product_id = f.product_id
    JOIN station_conductor c      ON m.product_id = c.product_id
    JOIN station_laser l          ON m.product_id = l.product_id
    JOIN station_tapitat t        ON m.product_id = t.product_id
    WHERE m.product_type = 'PT77'
)
SELECT
    COUNT(*)                                                            AS total_pt77,
    SUM(CASE WHEN molding='OK' AND quality_check='PASS' AND foaming='OK'
              AND conductor='OK' AND laser='OK' AND tapitat='OK'
             THEN 1 ELSE 0 END)                                         AS all_stations_ok,
    ROUND(100.0 * SUM(CASE WHEN molding='OK' AND quality_check='PASS' AND foaming='OK'
                            AND conductor='OK' AND laser='OK' AND tapitat='OK'
                           THEN 1 ELSE 0 END) / COUNT(*), 2)            AS first_pass_yield_pct
FROM pt77_full
"""
q(sql, rows=-1)

### Q9 — Material stock alert: what needs reordering?
> *Ce materiale am? — the batch-start question from the SIC.*

In [ ]:
sql = """
SELECT
    material_type,
    supplier,
    grade,
    unit,
    ROUND(quantity_in_stock,  1)  AS qty_in_stock,
    ROUND(reorder_threshold,  1)  AS reorder_threshold,
    ROUND(quantity_in_stock / NULLIF(reorder_threshold, 0), 2) AS stock_ratio,
    CASE
        WHEN quantity_in_stock <= reorder_threshold       THEN 'REORDER NOW'
        WHEN quantity_in_stock <= reorder_threshold * 1.5 THEN 'LOW STOCK'
        ELSE                                                   'OK'
    END                          AS stock_status
FROM materials_stock
ORDER BY stock_ratio ASC
"""
q(sql, rows=20)

---
## 5. Window Functions

Compute aggregates **without collapsing rows**.

```sql
RANK()     OVER (PARTITION BY group  ORDER BY metric DESC)
SUM(val)   OVER (                    ORDER BY time_col)    -- running total
LAG(val)   OVER (ORDER BY time_col)                        -- previous row
```

### Q10 — Rank operators by scrap rate within each shift
> *`PARTITION BY shift` resets the rank counter per shift.*

In [ ]:
sql = """
WITH stats AS (
    SELECT
        shift, operator_id,
        COUNT(*)                                                              AS total,
        SUM(CASE WHEN cycle_result = 'SCRAP' THEN 1 ELSE 0 END)               AS scraps,
        ROUND(100.0 * SUM(CASE WHEN cycle_result = 'SCRAP' THEN 1 ELSE 0 END)
                    / COUNT(*), 2)                                             AS scrap_pct
    FROM station_molding
    GROUP BY shift, operator_id
    HAVING COUNT(*) >= 50
)
SELECT
    shift, operator_id, total, scraps, scrap_pct,
    RANK() OVER (PARTITION BY shift ORDER BY scrap_pct DESC) AS rank_in_shift
FROM stats
ORDER BY shift, rank_in_shift
"""
q(sql, rows=15)

### Q11 — Cumulative Mg ingot consumption over time
> *Running total: `SUM(...) OVER (ORDER BY day)` — no GROUP BY needed.*

In [ ]:
sql = """
SELECT
    CAST(timestamp AS DATE)            AS day,
    ROUND(SUM(quantity_used), 3)        AS daily_mg_kg,
    ROUND(SUM(SUM(quantity_used)) OVER (
        ORDER BY CAST(timestamp AS DATE)
    ), 3)                               AS cumulative_mg_kg
FROM materials_log
WHERE material_type    = 'Mg_ingot'
  AND transaction_type = 'OUT'
GROUP BY CAST(timestamp AS DATE)
ORDER BY day
LIMIT 20
"""
q(sql, rows=-1)

### Q12 — Batch-over-batch foam deviation: is drift building up?
> *`LAG()` shows how today's batch average compares to the previous one.*

In [ ]:
sql = """
WITH batch_avg AS (
    SELECT
        batch_id, product_type,
        COUNT(*)                    AS units,
        ROUND(AVG(foam_volume_ml), 3) AS avg_foam_ml
    FROM station_foaming
    GROUP BY batch_id, product_type
)
SELECT
    batch_id, product_type, units, avg_foam_ml,
    ROUND(
        avg_foam_ml - LAG(avg_foam_ml) OVER (
            PARTITION BY product_type ORDER BY batch_id
        ), 3
    ) AS delta_vs_prev_batch
FROM batch_avg
ORDER BY product_type, batch_id
LIMIT 20
"""
q(sql, rows=-1)

---
## 6. Quality Validation — Checking SIC Rules with SQL

The SIC document defines pass/fail rules. SQL lets us verify them at scale.

> **Principle:** If you can write the rule in plain English, you can write it as a SQL `CASE WHEN`.

### Q13 — Validate SIC Rule: PT55 foam = 8.0 ml ±0.5 ml
> *Every PT55 unit outside 7.5–8.5 ml is a spec violation.*

In [ ]:
sql = """
SELECT
    COUNT(*)                                                             AS total_pt55,
    SUM(CASE WHEN foam_volume_ml BETWEEN 7.5 AND 8.5 THEN 1 ELSE 0 END) AS in_spec,
    SUM(CASE WHEN foam_volume_ml < 7.5              THEN 1 ELSE 0 END)  AS underfill_violations,
    SUM(CASE WHEN foam_volume_ml > 8.5              THEN 1 ELSE 0 END)  AS overfill_violations,
    ROUND(100.0 * SUM(CASE WHEN foam_volume_ml BETWEEN 7.5 AND 8.5
                           THEN 1 ELSE 0 END) / COUNT(*), 2)            AS in_spec_pct,
    ROUND(AVG(foam_volume_ml), 4)                                        AS grand_avg_ml,
    ROUND(STDDEV(foam_volume_ml), 4)                                     AS stddev_ml
FROM station_foaming
WHERE product_type = 'PT55'
"""
q(sql, rows=-1)

### Q14 — Validate SIC Rule: conductor resistance < 2.5 Ω
> *Break down by product type and result category.*

In [ ]:
sql = """
SELECT
    product_type,
    result,
    COUNT(*)                       AS count,
    ROUND(AVG(resistance_ohm), 3)   AS avg_ohm,
    ROUND(MIN(resistance_ohm), 3)   AS min_ohm,
    ROUND(MAX(resistance_ohm), 3)   AS max_ohm
FROM station_conductor
GROUP BY product_type, result
ORDER BY product_type, result
"""
q(sql, rows=-1)

### Q15 — Validate SIC Rule: molding temperature 180–220 °C
> *Temperature out of spec per product type and shift — where is the drift?*

In [ ]:
sql = """
SELECT
    product_type,
    shift,
    COUNT(*)                                                                  AS total,
    SUM(CASE WHEN temperature_c < 180                    THEN 1 ELSE 0 END)   AS below_180,
    SUM(CASE WHEN temperature_c > 220                    THEN 1 ELSE 0 END)   AS above_220,
    SUM(CASE WHEN temperature_c BETWEEN 180 AND 220      THEN 1 ELSE 0 END)   AS in_spec,
    ROUND(AVG(temperature_c), 2)                                               AS avg_temp_c,
    ROUND(100.0 * SUM(CASE WHEN temperature_c BETWEEN 180 AND 220
                           THEN 1 ELSE 0 END) / COUNT(*), 2)                  AS in_spec_pct
FROM station_molding
GROUP BY product_type, shift
ORDER BY product_type, shift
"""
q(sql, rows=-1)

### Q16 — Validate SIC Rule: PT77 laser power 80–120 W
> *Also check: does grip conductivity failure correlate with power being out of spec?*

In [ ]:
sql = """
SELECT
    grip_conductivity_test,
    COUNT(*)                                                               AS total,
    ROUND(AVG(laser_power_w), 2)                                           AS avg_power_w,
    SUM(CASE WHEN laser_power_w BETWEEN 80 AND 120 THEN 1 ELSE 0 END)      AS power_in_spec,
    SUM(CASE WHEN laser_power_w < 80  THEN 1 ELSE 0 END)                   AS power_below_80,
    SUM(CASE WHEN laser_power_w > 120 THEN 1 ELSE 0 END)                   AS power_above_120
FROM station_laser
GROUP BY grip_conductivity_test
"""
q(sql, rows=-1)

---
## 7. Challenge Questions

These have no answers. Write the SQL yourself.

---

**Challenge 1**  
> *Find all product_ids where the weight at QC is OUTSIDE the SIC limits for their product type.*  
> Hint: PT55 = 360–400 g, PT66 = 400–440 g, PT77 = 440–480 g.

---

**Challenge 2**  
> *Which operator + mold_tool_id combination produces the most scraps at molding?*  
> Use a 2-column `GROUP BY`.

---

**Challenge 3**  
> *For each product type, calculate the percentage of units that make it from molding to tapitat without a single SCRAP result.*  
> Use JOINs across all relevant stations.

---

**Challenge 4**  
> *Show the daily foam consumption (kg) vs daily leather consumption (dm²) side by side.*  
> Use `materials_log` with two aggregations filtered by `material_type`.

---

**Challenge 5**  
> *Find the top 3 batches where the average QC weight deviates most from the SIC target for their product type.*  
> Use a CTE to calculate expected weight, then measure deviation.